# 🚀 ArquiSys AI - Ollama en Google Colab

Este notebook ejecuta Ollama con qwen2.5:3b y expone el endpoint via ngrok.

## Instrucciones:
1. Runtime → Change runtime type → GPU (T4)
2. Ejecutar todas las celdas en orden
3. Al final copiar la URL de ngrok
4. Actualizar .env en tu sistema local

In [ ]:
#@title 1. Instalar todo desde cero
import subprocess
import os
import time

# Instalar zstd y Ollama
print("📦 Instalando dependencias...")
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, capture_output=True)
print("📥 Instalando Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True)
print("✅ Ollama instalado")

# Buscar ruta de Ollama
result = subprocess.run("which ollama", shell=True, capture_output=True, text=True)
OLLAMA_PATH = result.stdout.strip() or "/usr/local/bin/ollama"
print(f"📍 Ollama encontrado en: {OLLAMA_PATH}")

In [ ]:
#@title 2. Iniciar Ollama
import subprocess
import time

OLLAMA_PATH = "/usr/local/bin/ollama"

# Iniciar Ollama en background
subprocess.Popen([OLLAMA_PATH, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
time.sleep(5)
print("✅ Ollama iniciado en puerto 11434")

# Verificar que funciona
result = subprocess.run([OLLAMA_PATH, "list"], capture_output=True, text=True)
print("📋 Modelos disponibles:")
print(result.stdout if result.stdout else "Ninguno aún")

In [ ]:
#@title 3. Descargar modelo qwen2.5:3b
import subprocess

OLLAMA_PATH = "/usr/local/bin/ollama"

print("📥 Descargando modelo qwen2.5:3b (~2GB)...")
result = subprocess.run([OLLAMA_PATH, "pull", "qwen2.5:3b"], capture_output=True, text=True, timeout=600)
if result.returncode == 0:
    print("✅ Modelo descargado!")
else:
    print("⚠️ Estado de descarga:", result.returncode)

# Verificar
result = subprocess.run([OLLAMA_PATH, "list"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
#@title 4. Instalar ngrok y configurar token
import subprocess
import os

if not os.path.exists("/usr/local/bin/ngrok"):
    print("📥 Instalando ngrok...")
    subprocess.run("wget -q https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip -O ngrok.zip", shell=True, capture_output=True)
    subprocess.run("unzip -o ngrok.zip", shell=True, capture_output=True)
    subprocess.run("mv ngrok /usr/local/bin/", shell=True, capture_output=True)
    print("✅ ngrok instalado")
else:
    print("✅ ngrok ya instalado")

# Configurar token
subprocess.run("mkdir -p ~/.ngrok2", shell=True, capture_output=True)
ngrok_dir = os.path.expanduser("~/.ngrok2")
with open(os.path.join(ngrok_dir, "ngrok.yml"), "w") as f:
    f.write("authtoken: cr_3D2ftOD9E86bIfdoD8Z3eYEDplc")
print("✅ Token de ngrok configurado")

In [ ]:
#@title 5. Iniciar ngrok y obtener URL
import subprocess
import time
import urllib.request
import json

# Iniciar ngrok
subprocess.Popen(["/usr/local/bin/ngrok", "http", "11434", "--host-header", "rewrite"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
time.sleep(10)

# Obtener URL
try:
    response = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=10)
    data = json.loads(response.read())
    public_url = data['tunnels'][0]['public_url']
    print("="*50)
    print("🎉 URL PÚBLICA DE OLLAMA:")
    print(public_url)
    print("="*50)
    print("\n📝 COPIA ESTA URL y actualizá tu .env:")
    print(f"OLLAMA_BASE_URL={public_url}")
    print("\n⚠️ Esta URL cambia al reiniciar el notebook")
except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Esperando más tiempo...")
    time.sleep(15)
    try:
        response = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=10)
        data = json.loads(response.read())
        print("🎉 URL:", data['tunnels'][0]['public_url'])
    except:
        print("❌ No se pudo obtener la URL")

In [ ]:
#@title 6. Verificar funcionamiento y probar Ollama
import urllib.request
import json
import time

try:
    response = urllib.request.urlopen("http://localhost:11434/api/tags", timeout=15)
    models = json.loads(response.read())
    print("✅ Modelos disponibles:")
    for m in models.get('models', []):
        print(f"   - {m['name']}")
except Exception as e:
    print(f"⚠️ {e}")

# Prueba rápida con el modelo
print("\n🧪 Probando el modelo...")
try:
    import urllib.request
    import json
    
    payload = json.dumps({
        "model": "qwen2.5:3b",
        "prompt": "Responde solo con 'Hola, funciono!' en español",
        "stream": False
    }).encode()
    
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        result = json.loads(resp.read())
        print("✅ Respuesta del modelo:")
        print(result.get("response", "Sin respuesta"))
    print("\n🎉 Ollama funcionando perfectamente en la nube!")
except Exception as e:
    print(f"⚠️ Error en prueba: {e}")

In [ ]:
#@title 7. Crear mini interfaz web
print("📝 Instrucciones para usar la interfaz en tu PC:")
print("="*60)

# Obtener URL actual
try:
    response = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=10)
    data = json.loads(response.read())
    public_url = data['tunnels'][0]['public_url']
    print(f"🌐 URL de Ollama en la nube:")
    print(public_url)
    print()
except:
    print("⚠️ No se pudo obtener la URL")
    public_url = "TU-URL-AQUI"

print("""
╔══════════════════════════════════════════════════════════════╗
║                    🎉 CONFIGURACIÓN COMPLETA                   ║
╠══════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. En tu PC, editá el archivo .env:                              ║
║                                                                     ║
║       USE_OLLAMA_FOR_ANALYST=true                                 ║
║       OLLAMA_MODEL=qwen2.5:3b                                      ║
║       OLLAMA_BASE_URL={public_url}                              ║
║                                                                      ║
║  2. Ejecutá en tu terminal:                                       ║
║       py -m streamlit run scripts/ui_app.py                      ║
║                                                                      ║
║  3. Abrí en tu navegador:                                          ║
║       http://localhost:8501                                       ║
║                                                                      ║
║  ⚠️ Recordá: Si reiniciás el notebook, la URL cambia.            ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════╝
""")